# Is MoM-BH\*-1 near its Eddington limit? Rerun with the corrected fits (stage 5, version 3)

**Why a rerun:** on 8 October 2026 Olivia Curtis told you that her first public table had a flaw. The fits for her supplementary objects, MoM-BH\*-1 included, left the Balmer break unmasked, which made them come out too hot. She has replaced them with masked fits. Her corrected fit for MoM-BH\*-1: **4,950 K, log g = −2.77, and φ = 224 (34–653)**.

Rohan Naidu's reply adds two points:
- the interpretation of the two side components of Hβ is still uncertain and their signal-to-noise ratio is low (his team gets follow-up data over the next year), so the gas-speed check is now only a side check;
- the escape-speed argument gives an upper limit on the mass, which means a *lower* limit on the Eddington ratio.

**The question is the same as before:** stage 4 *assumed* that MoM-BH\*-1 shines at about its Eddington limit, which gave a total mass of about 1.7 million Suns. Is that allowed?

**Two published ways to weigh it without that assumption** (both from Naidu et al. 2026b, arXiv:2606.30711):
- **Surface gravity:** M = g R² / G, with the gravity g read from the shape of the spectrum.
- **Gas speed:** gas escaping from an object moves at least at the escape speed, √(2GM/R). So the speed v of outflowing gas gives M ≤ v² R / (2G).

Naidu et al. treat both as **upper limits** on the mass, because they picture the glowing surface as an outflowing wind.

**Two Eddington ratios. Don't mix them up** (this is what puzzled Rohan Naidu):
- for the **whole object** (black hole + envelope), the classic quasi-star model needs about 1 (0.76–1.58);
- for the **black hole alone**, it is about 170 for your 11,000-Sun black hole. The classic model expects this: the envelope, not the black hole, sets the limit.

Everything in this notebook is about the whole object, unless it says "black hole alone".

**The steps:**
- Step 1: your numbers.
- Step 2: surface gravity, with the corrected table.
- Step 3: gas speed (a side check now).
- Step 4: both checks side by side.
- Step 5: how robust is your black-hole mass?
- Step 6: the figure for your note.

**How to use:** upload this notebook to Colab and run the grey code cells from top to bottom with **Shift + Enter** (or *Runtime → Run all*). It takes about a minute, and it downloads the corrected table itself.

## Step 1 — Set up: constants and your numbers

This cell stores the constants, your stage 4 results, your measurements, and the published gas speeds. It is the same as in version 2, plus one line: the Eddington ratio of the black hole alone.

**Check the first block against your own stage 4 output** (Step 6, the block called "COMBINED (headline)"). If yours differ, change them here.

**The published speeds:** Naidu et al. (2026a) fitted the Hβ line with one emission line and three absorption components: one at the line centre and one on each side, at **−1,532 (+345/−113) km/s** and **+1,556 (+232/−1,378) km/s**. Minus means gas moving towards us (an outflow); plus means gas moving away from us (infall).

**Error bars in this notebook:** the function `spread()` draws 400,000 possible values of a measurement inside its error bar. Any quantity calculated from those draws then gets its own range automatically (the middle value and the range holding the middle 68%).

In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar, brentq

# --- constants (cgs units) ---
G, c, sigma_sb = 6.674e-8, 2.998e10, 5.6704e-5
Msun, Lsun, AU, KM = 1.989e33, 3.828e33, 1.496e13, 1e5
KAPPA_ES = 0.34                       # how strongly light pushes on ionised gas, cm^2/g (as in stages 2-4)

def eddington_mass(L):
    # the mass (in Suns) whose Eddington limit equals the luminosity L (erg/s)
    return KAPPA_ES * L / (4*np.pi*G*c) / Msun

def radius(L, T):
    # radius of the glowing surface (cm) from the Stefan-Boltzmann law
    return np.sqrt(L / (4*np.pi*sigma_sb*T**4))

# --- your stage 4 results (Step 6, "COMBINED" block): low, middle, high of the 68% range, in Suns ---
M_TOTAL = (1.25e6, 1.70e6, 2.35e6)    # total mass
M_BH = (6354, 11446, 22895)           # black hole

# --- your measurements ---
L_OBS, SIG_LOGL = 2.9e44, 0.1         # luminosity (erg/s), +/- 0.1 in log
T_OBS, SIG_T = 5200, 300              # surface temperature, K

# --- published by Naidu et al. (2026a): speeds of the two side components of H-beta, km/s: low, middle, high ---
V_BLUE = (1187, 1532, 1645)           # gas moving towards us:   -1,532 (+345/-113)
V_RED = (178, 1556, 1788)             # gas moving away from us: +1,556 (+232/-1,378)

# --- random draws inside the error bars ---
N_DRAW = 400_000
rng = np.random.default_rng(7)        # fixed seed: you get the same numbers every run

def spread(low, mid, high):
    # draws of a quantity whose 68% range is low-high around mid (the two sides can differ)
    z = rng.standard_normal(N_DRAW)
    return np.where(z < 0, mid + z*(mid - low), mid + z*(high - mid))

def middle_and_range(x):
    lo, mid, hi = np.percentile(x, [16, 50, 84])
    return mid, lo, hi

T_draw = spread(T_OBS - SIG_T, T_OBS, T_OBS + SIG_T)
L_draw = L_OBS * 10**spread(-SIG_LOGL, 0.0, SIG_LOGL)
R_draw = radius(L_draw, T_draw)                      # cm
v_draw = np.abs(spread(*V_BLUE))                     # km/s

mid, lo, hi = middle_and_range(R_draw/AU)
print(f"Your total mass:   {M_TOTAL[1]:.2e} Suns (range {M_TOTAL[0]:.2e} - {M_TOTAL[2]:.2e})")
print(f"Your black hole:   {M_BH[1]:,} Suns (range {M_BH[0]:,} - {M_BH[2]:,})")
print(f"Your radius:       {mid:,.0f} AU (range {lo:,.0f} - {hi:,.0f})")
print(f"Eddington mass for your luminosity: {eddington_mass(L_OBS):.2e} Suns")
print(f"Black hole alone:  shines {eddington_mass(L_OBS)/M_BH[1]:.0f} times above its own limit "
      f"(the whole object: about 1, by assumption)")

# --- plot style and colours (same as stages 2-4) ---
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": "#0b0b0b", "axes.titlecolor": "#0b0b0b", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "font.size": 11, "legend.frameon": False, "legend.fontsize": 9.5})
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, MUTED, ZONE, SURFACE = "#0b0b0b", "#52514e", "#898781", "#ebeae5", "#fcfcfb"

## Step 2 — Surface gravity, with the corrected table

Curtis et al. (2026) fitted star-like atmosphere models to 87 little red dots and published the results. The cell downloads the **corrected** table and works out, for each object:
- its **radius**, from their luminosity and temperature;
- its **mass from surface gravity**, M = g R² / G;
- its **Eddington ratio** φ: how many times above its Eddington limit it shines (their own column).

For MoM-BH\*-1 it also combines **their gravity with your surface** (your temperature and luminosity).

It saves a copy of the table as **curtis_table_corrected.csv**. Keep it with your notebooks: it records exactly what you used.

**Check first:** the cell prints **PASS** if the table has 87 objects and MoM-BH\*-1's row is the corrected fit (about 4,950 K). If it prints FAIL, the table has changed again: tell me.

**Expect:** MoM-BH\*-1 now shines about 220 times above its limit with their surface (range about 34–650), and its gravity mass is about 7,600 Suns (range about 2,600–60,000). The typical object shines about 15 times above its limit.

**The plot:** luminosity across, gravity mass up. The grey lines mark objects shining exactly at their limit, 10 times above it, and 100 times above it. The open square shows the mass a classic quasi-star as bright as MoM-BH\*-1 would need.

In [ ]:
URL = "https://raw.githubusercontent.com/o-curtis/lrd-quasistar-fits/main/summary/photospheric_parameters.csv"
table = pd.read_csv(URL)
table.to_csv("curtis_table_corrected.csv", index=False)
print(f"Table downloaded on {datetime.date.today()} and saved as curtis_table_corrected.csv\n")

main = table[table["family"].isin(["primary", "supplementary/prism", "supplementary/desi",
                                   "supplementary/grating"])].drop_duplicates("target").copy()
main["L"] = 10**main.logL_50 * Lsun                                    # luminosity, erg/s
main["R_au"] = radius(main.L, main.teff_50) / AU                       # radius, AU
main["mass"] = 10**main.logg_50 * (main.R_au*AU)**2 / G / Msun         # mass from gravity, Suns
is_mom = (main["target"] == "MoMBH1").values
mom = main[is_mom].iloc[0]
others = main[~is_mom]

# MoM-BH*-1's gravity mass with THEIR surface: low, middle, high
M_GRAV = tuple(10**mom[f"logg_{p}"] * (mom.R_au*AU)**2 / G / Msun for p in ("16", "50", "84"))

# ... and their gravity with YOUR surface (separate random draws, so Steps 3-4 match version 2)
z = np.random.default_rng(11).standard_normal(N_DRAW)
logg_draw = np.where(z < 0, mom.logg_50 + z*(mom.logg_50 - mom.logg_16), mom.logg_50 + z*(mom.logg_84 - mom.logg_50))
mass_mine_draw = 10**logg_draw * R_draw**2 / G / Msun
ratio_mine_draw = eddington_mass(L_draw) / mass_mine_draw
M_GRAV_MINE = middle_and_range(mass_mine_draw)
RATIO_GRAV_MINE = middle_and_range(ratio_mine_draw)

print(f"{len(main)} little red dots loaded")
print(f"Typical little red dot: shines {main.phi_50.median():.0f} times above its limit\n")
print(f"MoM-BH*-1, their corrected fit: {mom.teff_50:,.0f} K ({mom.teff_16:,.0f} - {mom.teff_84:,.0f}), "
      f"{mom.L:.1e} erg/s, radius {mom.R_au:,.0f} AU, log g = {mom.logg_50:.2f}")
print(f"   shines {mom.phi_50:.0f} times above its limit (range {mom.phi_16:.0f} - {mom.phi_84:.0f})")
print(f"   gravity mass {M_GRAV[1]:,.0f} Suns (range {M_GRAV[0]:,.0f} - {M_GRAV[2]:,.0f})")
print(f"MoM-BH*-1, their gravity with your surface ({T_OBS:,} K, {L_OBS:.1e} erg/s):")
print(f"   shines {RATIO_GRAV_MINE[0]:.0f} times above its limit (range {RATIO_GRAV_MINE[1]:.0f} - {RATIO_GRAV_MINE[2]:.0f})")
print(f"   gravity mass {M_GRAV_MINE[0]:,.0f} Suns (range {M_GRAV_MINE[1]:,.0f} - {M_GRAV_MINE[2]:,.0f})\n")
print(f"A classic quasi-star as bright as this needs about {eddington_mass(L_OBS):.1e} Suns in all: "
      f"{eddington_mass(L_OBS)/M_GRAV_MINE[0]:.0f} times more than the gravity allows.")
print(f"Chance that the gravity allows the classic range (below 1.58): {100*np.mean(ratio_mine_draw < 1.58):.1f}%\n")
print(f"Other objects further above their limit than MoM-BH*-1 (middle values): {np.mean(others.phi_50 > mom.phi_50)*100:.0f}%")
print(f"Objects that could be within the classic range inside their error bar: "
      f"{int((others.phi_16 < 1.58).sum())} of the other {len(others)}; MoM-BH*-1: {'yes' if mom.phi_16 < 1.58 else 'no'}")
odd = main[(main.logL_50 > 13) | (main.logL_50 < 8)]
print(f"Fits with an implausible luminosity (treat with care): {', '.join(odd.target) if len(odd) else 'none'}")
ok = (len(main) == 87) and (4700 < mom.teff_50 < 5200) and (-3.0 < mom.logg_50 < -2.5)
print(f"\n{'PASS' if ok else 'FAIL'}  87 objects loaded, and MoM-BH*-1's row is the corrected fit (about 4,950 K, log g about -2.77)")

fig, ax = plt.subplots(figsize=(8.5, 5.6))
ax.set_axisbelow(True)
L_line = np.logspace(41, 47, 50)
for ratio, word in [(1, "at its limit"), (10, "10× above"), (100, "100× above")]:
    ax.plot(L_line, eddington_mass(L_line)/ratio, color=MUTED, lw=1)
    ax.text(2.6e45, eddington_mass(2.6e45)/ratio*1.35, word, color=INK2, fontsize=9.5, ha="right",
            rotation=33, rotation_mode="anchor")
ax.plot(others.L, others.mass, "o", color=BLUE, ms=6.5, alpha=0.75, mec=SURFACE, mew=0.8,
        label="little red dots (Curtis et al. 2026, corrected)")
ax.plot([mom.L]*2, [M_GRAV[2], eddington_mass(mom.L)], color=ORANGE, lw=1, ls=":")
ax.errorbar(mom.L, M_GRAV[1], yerr=[[M_GRAV[1]-M_GRAV[0]], [M_GRAV[2]-M_GRAV[1]]], fmt="s", color=ORANGE,
            ms=9, mec=SURFACE, mew=1.2, lw=2, label="MoM-BH*-1, from its surface gravity")
ax.plot(mom.L, eddington_mass(mom.L), "s", ms=9, mfc=SURFACE, mec=ORANGE, mew=1.8,
        label="MoM-BH*-1, if it were a classic quasi-star")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlim(1e42, 3e45); ax.set_ylim(1e2, 1e8)
outside = int(((main.L < 1e42) | (main.L > 3e45) | (main.mass < 1e2) | (main.mass > 1e8)).sum())
ax.set_xlabel("Luminosity (erg/s)")
ax.set_ylabel("Mass from surface gravity (Suns)")
ax.set_title("With the corrected fits, MoM-BH*-1 sits far above its limit", loc="left")
ax.legend(loc="upper left")
plt.tight_layout(); fig.savefig("s5_step2_gravity_v3.png", dpi=150); plt.show()
print(f"({outside} of the {len(main)} objects fall outside the plotted area.)")

**What it means:** with the corrected fit, MoM-BH\*-1's surface gravity is low. Everything inside its glowing surface weighs only about 7,600–7,800 Suns (up to about 60,000). A classic quasi-star this bright needs about 2 million Suns there, a few hundred times more. So the gravity now says MoM-BH\*-1 shines about 200–300 times above its limit, and the chance that it allows the classic range is below 1%.

**Be careful how you say this:**
- **Gravity from a prism spectrum is only partly constrained.** Curtis et al. show (their Section 4.4) that prism-only fits gain some information over their starting assumptions, but cannot pin the gravity down without water absorption features. Her fit still puts all of its probability above 1 and 94% above 10.
- **The "chance" above treats her 16–84% range as a bell curve.** Her own posterior has nothing below 1, so the real chance is smaller still.
- **In the discovery team's picture the gravity mass is an upper limit**, because a wind makes the gravity look stronger than it is (Naidu et al. 2026b). That would put MoM-BH\*-1 even further above its limit.
- **Your surface and theirs now agree:** 5,200 ± 300 K and 2.9 × 10⁴⁴ erg/s against their 4,950 K and 2.5 × 10⁴⁴ erg/s.

## Step 3 — Gas speed (now a side check)

**Rohan Naidu's caveat first:** the interpretation of the two side components of Hβ is uncertain, and their signal-to-noise ratio is low at the moment. His team has not seen them in other objects and gets follow-up data over the next year. So treat this step as a side check, and say so if you use it.

**The physics:** to leave an object of mass M from radius R, gas needs the escape speed, **v = √(2GM/R)**. So gas escaping at speed v tells you **M ≤ v² R / (2G)**: an upper limit on the mass, which is a **lower** limit on the Eddington ratio.

**What the cell does** (unchanged from version 2):
- the escape speed your near-limit envelope would have. **Expect about 1,380 km/s.**
- the measured outflow speed (1,532 km/s) as an upper limit on the mass. **Expect about 2.0 million Suns**, so the object shines **at least** about once its limit.
- the same if the gas is a wind at 3–5 times the escape speed (Naidu et al. 2026b).

In [ ]:
# the envelope stage 4 assumed: total mass = Eddington mass / f, with f anywhere between 0.76 and 1.58
f_draw = rng.uniform(0.76, 1.58, N_DRAW)
M_near = eddington_mass(L_draw) / f_draw
V_PRED = middle_and_range(np.sqrt(2*G*M_near*Msun / R_draw) / KM)

# upper limit on the mass from the outflowing gas, and the Eddington ratio that goes with it
M_SPEED = middle_and_range((v_draw*KM)**2 * R_draw / (2*G) / Msun)
RATIO_SPEED = middle_and_range(eddington_mass(L_draw) / ((v_draw*KM)**2 * R_draw / (2*G) / Msun))

# the escape speed if it shone 5-50 times above its limit
V_LIGHT = tuple(np.sqrt(2*G*eddington_mass(L_OBS)/k*Msun / radius(L_OBS, T_OBS)) / KM for k in (50, 5))

print(f"Escape speed of your near-limit envelope:  {V_PRED[0]:,.0f} km/s  (range {V_PRED[1]:,.0f} - {V_PRED[2]:,.0f})")
print(f"Measured, gas moving towards us:           {V_BLUE[1]:,} km/s  (range {V_BLUE[0]:,} - {V_BLUE[2]:,})")
print(f"Measured, gas moving away from us:         {V_RED[1]:,} km/s  (range {V_RED[0]:,} - {V_RED[2]:,})")
print(f"Escape speed if 5-50 times above its limit: {V_LIGHT[0]:,.0f} - {V_LIGHT[1]:,.0f} km/s\n")
print(f"Upper limit on the mass from the outflow:  {M_SPEED[0]:.2e} Suns  (range {M_SPEED[1]:.2e} - {M_SPEED[2]:.2e})")
print(f"So it shines at least {RATIO_SPEED[0]:.1f} times its Eddington limit (range {RATIO_SPEED[1]:.1f} - {RATIO_SPEED[2]:.1f})")
print(f"If the outflow is a wind at 3-5 times the escape speed: mass {M_SPEED[0]/25:.1e} - {M_SPEED[0]/9:.1e} Suns, "
      f"{RATIO_SPEED[0]*9:.0f} - {RATIO_SPEED[0]*25:.0f} times above its limit")

rows = ["Escape speed of your near-limit envelope", "Measured: gas moving towards us",
        "Measured: gas moving away from us", "Escape speed if 5–50× above its limit"]
fig, ax = plt.subplots(figsize=(9.5, 3.8))
ax.set_axisbelow(True)
ax.barh(3, V_PRED[2]-V_PRED[1], left=V_PRED[1], height=0.24, color=BLUE)
ax.plot(V_PRED[0], 3, "o", color=SURFACE, ms=6)
ax.errorbar(V_BLUE[1], 2, xerr=[[V_BLUE[1]-V_BLUE[0]], [V_BLUE[2]-V_BLUE[1]]], fmt="o", color=ORANGE, ms=9,
            mec=SURFACE, mew=1.5, lw=2, capsize=4)
ax.errorbar(V_RED[1], 1, xerr=[[V_RED[1]-V_RED[0]], [V_RED[2]-V_RED[1]]], fmt="o", color=ORANGE, ms=9,
            mec=SURFACE, mew=1.5, lw=2, capsize=4)
ax.barh(0, V_LIGHT[1]-V_LIGHT[0], left=V_LIGHT[0], height=0.24, color=MUTED)
ax.set_yticks([3, 2, 1, 0]); ax.set_yticklabels(rows)
ax.tick_params(axis="y", length=0, labelcolor=INK)
ax.grid(axis="y", visible=False)
ax.set_xlim(0, 2000); ax.set_ylim(-0.6, 3.6)
ax.set_xlabel("Speed (km/s)")
ax.set_title("If escaping, the fastest gas only shows that it shines at least at its limit", loc="left")
plt.tight_layout(); fig.savefig("s5_step3_gas_speed.png", dpi=150); plt.show()

**What it means:** if the blueshifted component traces escaping gas, MoM-BH\*-1 shines **at least** about once its limit. That agrees with the gravity (about 200–300 times) but adds little: "at least 1" is far below what the gravity says.

**Cautions. Put them in anything you write:**
1. **Their interpretation is uncertain and their signal-to-noise ratio is low** (Naidu, private communication, October 2026).
2. **It is an upper limit on the mass, not a measurement.** Naidu et al. (2026b) argue that winds in these objects move at 3–5 times the escape speed. Then the ratio is 9–25 or more.
3. **The discovery paper does not read these features as moving gas.** Naidu et al. (2026a) say the symmetric Hβ profile makes inflows and outflows unlikely, and prefer one shell of gas, possibly with light scattering back and forth inside it.
4. **The infalling component can't help.** Its speed is only known to lie between 178 and 1,788 km/s.

## Step 4 — Both checks side by side

Everything is turned into one number: **how many times above its Eddington limit the whole object shines**. The cell does this for two versions of the glowing surface, which now agree:
- **yours:** 5,200 K and 2.9 × 10⁴⁴ erg/s (blackbody fit);
- **Curtis et al.'s corrected fit:** about 4,950 K and 2.5 × 10⁴⁴ erg/s.

The gas speed gives a *lower* limit. A classic quasi-star needs 0.76–1.58.

In [ ]:
# Curtis et al.'s corrected surface
T_c = spread(mom.teff_16, mom.teff_50, mom.teff_84)
L_c = 10**spread(mom.logL_16, mom.logL_50, mom.logL_84) * Lsun
R_c = radius(L_c, T_c)
RATIO_SPEED_C = middle_and_range(eddington_mass(L_c) / ((v_draw*KM)**2 * R_c / (2*G) / Msun))
RATIO_GRAV_C = (mom.phi_50, mom.phi_16, mom.phi_84)                   # their own column

print("How many times above its Eddington limit (whole object)?   your surface        Curtis et al.'s corrected surface")
print(f"  from surface gravity (their fit)                         {RATIO_GRAV_MINE[0]:5.0f}  ({RATIO_GRAV_MINE[1]:.0f} - {RATIO_GRAV_MINE[2]:.0f})"
      f"         {RATIO_GRAV_C[0]:5.0f}  ({RATIO_GRAV_C[1]:.0f} - {RATIO_GRAV_C[2]:.0f})")
print(f"  from the outflow speed, if escaping (at least)           {RATIO_SPEED[0]:5.1f}  ({RATIO_SPEED[1]:.1f} - {RATIO_SPEED[2]:.1f})"
      f"        {RATIO_SPEED_C[0]:5.1f}  ({RATIO_SPEED_C[1]:.1f} - {RATIO_SPEED_C[2]:.1f})")
print(f"  a classic quasi-star needs                                0.76 - 1.58")
print(f"  typical little red dot                                    about {main.phi_50.median():.0f} (Curtis et al.'s table); 5 - 50 (Sun et al. 2026)\n")

allowed = (RATIO_GRAV_MINE[1] < 1.58) or (RATIO_GRAV_C[1] < 1.58)
print(f"{'ALLOWED' if allowed else 'NOT ALLOWED'}: the corrected surface gravity {'reaches' if allowed else 'does not reach'} "
      f"the classic range inside its error bar.")
print(f"(For the black hole alone, your classic model gives about {eddington_mass(L_OBS)/M_BH[1]:.0f}. That is a different number.)")

**What it means, in one sentence:** with the corrected fits, a near-limit envelope is **not allowed**: the gravity puts MoM-BH\*-1 a few hundred times above its limit, far outside the classic range.

- The gas speed, if the side component traces escaping gas, only says "at least about 1". It doesn't help either way.
- **What none of this tells you:** how the mass is split between black hole and gas. These tests weigh everything inside the glowing surface. But since everything there weighs only about 7,600–7,800 Suns (up to about 60,000), **the black hole can't weigh more than that**.

## Step 5 — How robust is your black-hole mass?

Four tests, using the simulator itself.

1. **Reproduce stage 4.** **PASS** if the middle value and both ends of the range agree with your stage 4 numbers within 5%.
2. **Curtis et al.'s corrected surface** instead of yours.
3. **Drop the near-limit assumption.** What black hole gives 5,200 K if the object shines 1, 2, 3 or 5 times above its limit (so it is 1, 2, 3 or 5 times lighter)? Then the same at the ratios the corrected gravity implies (the low end of its range, and the middle). **Expect "NO SOLUTION" there:** the envelope is then so light that even the biggest black hole it can hold leaves the surface far hotter than 5,200 K.
4. **The temperature floor** in Ball et al.'s full model (unchanged).

In [ ]:
# ---- the simulator, copied from stage 4, Step 2 ----
GAMMA, n = 4/3, 3
MODEL_ERR = 0.10
PRIOR = {"log_total": (5.5, 8.0), "log_bh": (1.5, 7.5), "log_eta": (np.log10(0.03), np.log10(0.3)), "f": (0.76, 1.58)}

def lane_emden(xi, y):
    theta, phi = y
    return [-phi / xi**2, xi**2 * max(theta, 0.0)**n]

def at_surface(xi, y):
    return y[0]
at_surface.terminal, at_surface.direction = True, -1

def loaded_polytrope(xi0):
    phi_bh = xi0 / (2*n)
    sol = solve_ivp(lane_emden, [xi0, 100], [1.0, phi_bh + (2/3)*xi0**3], events=at_surface, rtol=1e-9, atol=1e-12)
    return phi_bh, sol.y_events[0][0][1], sol.t_events[0][0]

best = minimize_scalar(lambda x: -loaded_polytrope(x)[0]/loaded_polytrope(x)[1], bounds=(0.8, 1.2),
                       method="bounded", options={"xatol": 1e-7})
XI0_MAX, Q_MAX = best.x, -best.fun
_xi0 = np.geomspace(1e-7, XI0_MAX, 300)
_tab = np.array([loaded_polytrope(x) for x in _xi0])
_q, _phi_s, _xi1 = _tab[:, 0]/_tab[:, 1], _tab[:, 1], _tab[:, 2]

def simulate(log_total, log_bh, log_eta, f):
    M_tot, M_bh, eta = 10**log_total*Msun, 10**log_bh*Msun, 10**log_eta
    q = M_bh / M_tot
    valid = q < Q_MAX
    lq = np.log(np.clip(q, 1e-12, 0.999999*Q_MAX))
    phi_s = np.exp(np.interp(lq, np.log(_q), np.log(_phi_s)))
    xi1 = np.interp(lq, np.log(_q), _xi1)
    K = np.pi*G * (M_tot / (4*np.pi*phi_s))**(2/3)
    L = f * 4*np.pi*G*c*M_tot / KAPPA_ES
    A = (16*np.pi*eta/GAMMA) * (G*M_bh)**2 / np.sqrt(GAMMA*K)
    R = xi1 * np.sqrt(K/(np.pi*G)) * (L/A)**(-2/5)
    T = (L / (4*np.pi*sigma_sb*R**2))**0.25
    return np.log10(L), np.where(valid, np.log10(T), np.nan), valid

# ---- the inference, as in stage 4, Step 6, but with the measurements as inputs ----
CAL_SHARE = np.array([5e-4, 0.01, 0.02, 0.05, 0.1])
CAL_RATIO = np.array([1.10, 0.83, 0.73, 0.61, 0.68])

def weighted_range(x, w):
    order = np.argsort(x)
    cum = np.cumsum(w[order]); cum /= cum[-1]
    return np.interp([0.16, 0.5, 0.84], cum, x[order])

def infer(L, T, sig_T, sig_logL=0.1, N=2_000_000, seed=5):
    draw_rng = np.random.default_rng(seed)
    sig_logT = np.sqrt((sig_T / T / np.log(10))**2 + np.log10(1 + MODEL_ERR)**2)
    out = {}
    for label, calibrate in [("as is", False), ("calibrated", True)]:
        P = np.column_stack([draw_rng.uniform(*PRIOR[k], N) for k in PRIOR])
        logL, logT, valid = simulate(*P.T)
        if calibrate:
            logT = logT - np.log10(np.interp(P[:, 1] - P[:, 0], np.log10(CAL_SHARE), CAL_RATIO))
        chi2 = ((logL - np.log10(L))/sig_logL)**2 + ((logT - np.log10(T))/sig_logT)**2
        w = np.where(valid, np.exp(-0.5*np.nan_to_num(chi2, nan=1e9)), 0.0)
        out[label] = (P, w / w.sum())
    out["combined"] = (np.vstack([out["as is"][0], out["calibrated"][0]]),
                       np.concatenate([out["as is"][1], out["calibrated"][1]]))
    return out

def show(result, label):
    P, w = result["combined"]
    bh = 10**weighted_range(P[:, 1], w)
    tot = 10**weighted_range(P[:, 0], w)
    print(f"--- {label} ---")
    print(f"   black hole: {bh[1]:,.0f} Suns   (68% range {bh[0]:,.0f} - {bh[2]:,.0f})")
    print(f"   total mass: {tot[1]:.2e} Suns (68% range {tot[0]:.2e} - {tot[2]:.2e})")
    return bh, tot

# ---- test 1: reproduce stage 4 ----
print(f"Black-hole limit: {Q_MAX:.4f} (stage 4 gave 0.1015)\n")
res_mine = infer(L_OBS, T_OBS, SIG_T)
bh_mine, tot_mine = show(res_mine, f"test 1, your measurements: {T_OBS:,} K, {L_OBS:.1e} erg/s")
worst = max(abs(bh_mine[i]/M_BH[i] - 1) for i in range(3))
print(f"{'PASS' if worst < 0.05 else 'FAIL'}  matches your stage 4 black-hole mass within {100*worst:.1f}%\n")

# ---- test 2: Curtis et al.'s corrected surface ----
res_curtis = infer(mom.L, mom.teff_50, 0.5*(mom.teff_84 - mom.teff_16))
bh_curtis, tot_curtis = show(res_curtis, f"test 2, Curtis et al.'s corrected fit: {mom.teff_50:,.0f} K, {mom.L:.1e} erg/s")

# ---- test 3: drop the near-limit assumption ----
def surface_T(log_total, log_bh, eta, f):
    # surface temperature (K) of the model as is
    return 10**simulate(np.array([log_total]), np.array([log_bh]), np.array([np.log10(eta)]), np.array([f]))[1][0]

def black_hole_for(T, L, eta, f):
    # the black hole (Suns) that gives surface temperature T, for total mass = Eddington mass / f (model as is);
    # None if even the biggest black hole the envelope can hold leaves the surface hotter than T
    log_total = np.log10(eddington_mass(L) / f)
    top = log_total + np.log10(0.999*Q_MAX)
    if surface_T(log_total, top, eta, f) > T:
        return None
    return 10**brentq(lambda log_bh: surface_T(log_total, log_bh, eta, f) - T, 1.0, top)

print("\n--- test 3, the black hole that gives 5,200 K (model as is, eta = 0.1) ---")
for f in [1, 2, 3, 5]:
    bh = black_hole_for(T_OBS, L_OBS, 0.1, f)
    print(f"   {f} times above its limit: total {eddington_mass(L_OBS)/f:.1e} Suns, black hole {bh:,.0f} Suns "
          f"(share {bh/(eddington_mass(L_OBS)/f):.3f})")
print("   ... and at the ratios the corrected gravity implies:")
for f in [round(RATIO_GRAV_MINE[1]), round(RATIO_GRAV_MINE[0])]:
    total = eddington_mass(L_OBS) / f
    bh = black_hole_for(T_OBS, L_OBS, 0.1, f)
    if bh is None:
        T_top = surface_T(np.log10(total), np.log10(0.999*Q_MAX*total), 0.1, f)
        print(f"   {f} times above its limit: total {total:,.0f} Suns. NO SOLUTION: even the biggest black hole it can hold "
              f"({Q_MAX*total:,.0f} Suns) leaves the surface at {T_top:,.0f} K")
    else:
        print(f"   {f} times above its limit: total {total:,.0f} Suns, black hole {bh:,.0f} Suns (share {bh/total:.3f})")
b0 = black_hole_for(T_OBS, L_OBS, 0.1, 1.2)
print(f"   For comparison, eta = 0.03 gives {black_hole_for(T_OBS, L_OBS, 0.03, 1.2):,.0f} Suns and eta = 0.3 gives "
      f"{black_hole_for(T_OBS, L_OBS, 0.3, 1.2):,.0f} Suns.")
step = 1.02
print(f"   Scaling: black hole = {b0:,.0f} Suns x (T/5,200 K)^{np.log(black_hole_for(T_OBS*step, L_OBS, 0.1, 1.2)/b0)/np.log(step):.2f} "
      f"x (L/2.9e44)^{np.log(black_hole_for(T_OBS, L_OBS*step, 0.1, 1.2)/b0)/np.log(step):.2f} "
      f"x (eta/0.1)^{np.log(black_hole_for(T_OBS, L_OBS, 0.1*step, 1.2)/b0)/np.log(step):.2f} "
      f"x (f/1.2)^{np.log(black_hole_for(T_OBS, L_OBS, 0.1, 1.2*step)/b0)/np.log(step):.2f}")

# ---- test 4: the temperature floor in Ball et al.'s full model ----
T_FLOOR = 4500
print(f"\n--- test 4, the temperature floor ---")
print(f"   Ball et al.'s model levels off near {T_FLOOR:,} K. Your {T_OBS:,} +/- {SIG_T} K is {(T_OBS - T_FLOOR)/SIG_T:.1f} error bars above it.")
print(f"   Largest black hole a classic quasi-star of your total mass can hold: {Q_MAX*M_TOTAL[1]:.1e} Suns")

**What it means:**

- **Test 2:** with Curtis et al.'s corrected surface the black hole comes out at about 12,000 Suns, the same as with yours (test 1). Your result no longer depends on whose surface you use.
- **Test 3:** between 1 and 5 times above the limit the black hole hardly changes (it goes as the fourth root of the ratio). But at the ratios the corrected gravity implies, the classic model has **no solution at all**. So it fails twice: a hydrostatic envelope can't shine that far above its limit, and even if it could, it couldn't be this cool.
- **Test 4:** unchanged. Your temperature is only about two error bars above the floor in Ball et al.'s full model.

So the classic model's black hole, about 10,000 Suns, now answers "what if it *were* a classic quasi-star?", not "what is it?".

## Step 6 — The figure for your note

One figure, two panels, saved as **figure1_MoM-BH1_v3.png** (in the Colab file list on the left: right-click → Download).

- **Left (a):** how many times above its Eddington limit MoM-BH\*-1 shines (whole object). The top row is the corrected surface gravity; filled symbols use your surface, open symbols Curtis et al.'s. The middle row is the gas speed, if the Hβ side component traces escaping gas: a lower limit, so it is drawn with an arrow. The green band is what a classic quasi-star needs; the grey band is typical for these objects (Sun et al. 2026). The blue dots are the other 86 little red dots from the corrected table.
- **Right (b):** masses. The blue histogram is the black hole a classic quasi-star would need (Step 5, test 1, both versions combined). The black bar is everything inside the glowing surface according to the corrected gravity: the black hole can't weigh more than that. The orange band holds the published black-hole estimates.

The panels have no headline titles, because in a journal the caption carries the message.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1.25, 1]})
fig.patch.set_facecolor("white")
for ax in (ax1, ax2):
    ax.set_facecolor("white"); ax.set_axisbelow(True)

# ---- left: Eddington ratio of the whole object ----
labels = ["Surface gravity\n(Curtis et al. 2026, corrected)", "Blueshifted Hβ absorber, if it traces\nescaping gas (Naidu et al. 2026a)",
          f"{len(others)} other little red dots\n(Curtis et al. 2026, corrected)"]
ax1.axvspan(0.76, 1.58, ymin=0.33, ymax=1, color=AQUA, alpha=0.22, lw=0, zorder=0)
ax1.axvspan(5, 50, ymin=0.33, ymax=1, color=ZONE, lw=0, zorder=0)
ax1.text(np.sqrt(0.76*1.58), 2.62, "classic\nquasi-star", color=INK2, fontsize=9.5, ha="center", va="top")
ax1.text(np.sqrt(5*50), 2.62, "typical BH*\n(Sun et al. 2026)", color=INK2, fontsize=9.5, ha="center", va="top")

def point(ax, y, val, filled):
    mid, lo, hi = val
    ax.errorbar(mid, y, xerr=[[mid-lo], [hi-mid]], fmt="o", color=INK, ms=8.5, mfc=INK if filled else "white",
                mec=INK, mew=1.5, lw=2, capsize=4)

def lower_limit(ax, y, value, filled):
    ax.annotate("", xy=(value*4, y), xytext=(value, y),
                arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.6, mutation_scale=12))
    ax.plot(value, y, "o", color=INK, ms=8.5, mfc=INK if filled else "white", mec=INK, mew=1.5, zorder=5)

point(ax1, 2.12, RATIO_GRAV_MINE, True)
point(ax1, 1.88, RATIO_GRAV_C, False)
lower_limit(ax1, 1.12, RATIO_SPEED[0], True)
lower_limit(ax1, 0.88, RATIO_SPEED_C[0], False)
jitter = np.random.default_rng(1).uniform(-0.2, 0.2, len(others))
ax1.plot(others.phi_50, 0 + jitter, "o", color=BLUE, ms=5.5, alpha=0.65, mec="white", mew=0.6)
ax1.plot([others.phi_50.median()]*2, [-0.32, 0.32], color=INK, lw=1.5)
ax1.text(others.phi_50.median(), -0.42, "median", color=INK2, fontsize=9, ha="center", va="top")
ax1.plot([], [], "o", color=INK, ms=8.5, mfc=INK, label="my photosphere")
ax1.plot([], [], "o", color=INK, ms=8.5, mfc="white", mew=1.5, label="Curtis et al.'s corrected photosphere")
ax1.legend(loc="lower left", bbox_to_anchor=(0.0, 1.0), ncol=2, columnspacing=1.4, handletextpad=0.3)
ax1.set_xscale("log"); ax1.set_xlim(0.08, 3000); ax1.set_ylim(-0.85, 2.7)
ax1.set_yticks([2, 1, 0]); ax1.set_yticklabels(labels)
ax1.tick_params(axis="y", length=0, labelcolor=INK)
ax1.grid(axis="y", visible=False)
ax1.set_xlabel("Luminosity / Eddington luminosity (whole object)")
ax1.text(-0.62, 1.04, "(a)", transform=ax1.transAxes, fontsize=12, fontweight="bold", color=INK)

# ---- right: black-hole masses ----
# one posterior (the combined one), direct labels instead of a legend
bins = np.linspace(1.5, 7.5, 121)
P, w = res_mine["combined"]
h, edges = np.histogram(P[:, 1], bins=bins, weights=w, density=True)
peak = h.max()
ax2.axvspan(1e6, 1e7, color=ORANGE, alpha=0.2, lw=0)
ax2.text(np.sqrt(1e6*1e7), 1.93*peak, "published\nestimates", color=INK2, fontsize=9.5, ha="center", va="top")
ax2.fill_between(10**edges[:-1], h, step="post", color=BLUE, alpha=0.25, lw=0)
ax2.step(10**edges[:-1], h, where="post", color=BLUE, lw=1.8)
ax2.text(2.6e4, 0.92*peak, "black hole, if it were\na classic quasi-star", color=INK2, fontsize=9.5, ha="left", va="center")
y_cap = 1.45*peak
ax2.errorbar(M_GRAV_MINE[0], y_cap, xerr=[[M_GRAV_MINE[0]-M_GRAV_MINE[1]], [M_GRAV_MINE[2]-M_GRAV_MINE[0]]],
             fmt="o", color=INK, ms=8, mec="white", mew=1.2, lw=2, capsize=4)
ax2.text(M_GRAV_MINE[0], y_cap + 0.09*peak, "everything inside the photosphere\n(corrected surface gravity)",
         color=INK2, fontsize=9.5, ha="center", va="bottom")
ax2.set_xscale("log"); ax2.set_xlim(1e2, 3e7); ax2.set_ylim(0, 2.0*peak)
ax2.set_xlabel("Mass (M$_\\odot$)")
ax2.set_ylabel("Posterior probability density (per dex)")
ax2.text(0.0, 1.04, "(b)", transform=ax2.transAxes, fontsize=12, fontweight="bold", color=INK)

plt.tight_layout(); fig.savefig("figure1_MoM-BH1_v3.png", dpi=200, facecolor="white"); plt.show()
n_out = int(((others.phi_50 < 0.08) | (others.phi_50 > 3000)).sum())
print(f"({n_out} of the {len(others)} other little red dots fall outside the left panel's range.)")

# What the rerun tells you

- **Your surface is confirmed.** Curtis et al.'s corrected fit (4,950 K, 2.5 × 10⁴⁴ erg/s) agrees with yours (5,200 ± 300 K, 2.9 × 10⁴⁴ erg/s), and so does the luminosity of the discovery paper's CLOUDY model (about 3.2 × 10⁴⁴ erg/s, Extended Data Table 4).
- **A classic quasi-star is now strongly disfavoured** (Steps 2 and 4). The corrected gravity puts only about 7,600–7,800 Suns (up to about 60,000) inside the glowing surface, so MoM-BH\*-1 shines a few hundred times above its limit. A classic quasi-star needs about 1, and about 2 million Suns. The chance that the gravity allows that is below 1%.
- **Even ignoring that, the classic model has no solution at that mass** (Step 5, test 3).
- **The black hole is light either way.** If MoM-BH\*-1 were a classic quasi-star, its black hole would be about 10,000 Suns (Step 5, tests 1–2). If it isn't, the gravity still caps everything inside the glowing surface at about 7,600–7,800 Suns (up to about 60,000). Both are far below the published 1–10 million.
- **The gas speed is only a side check now** (Step 3).

**Numbers for your note** (copy them from your own output):
- Step 1: the black hole's own Eddington ratio.
- Step 2: MoM-BH\*-1's corrected values; its gravity mass and Eddington ratio for both surfaces; the chance of the classic range; the typical ratio of the 87.
- Step 3: the lower limit from the gas speed.
- Step 5: tests 1–3.

**How far to trust it:**
- The gravity comes from a prism-only fit, which Curtis et al. say is only partly constrained (their Section 4.4). Her fit still puts all of its probability above 1 and 94% above 10.
- If the glowing surface is a wind, the gravity mass is an upper limit (Naidu et al. 2026b). That makes the conclusion stronger, not weaker.
- Curtis's revised paper may already make this point; you asked her. Wait for her answer before writing the new version of the note.